# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [3]:
# 1. Cấu hình
import json
import os
import uuid

from kaggle_secrets import UserSecretsClient

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"
TRAIN_CONFIG = "train/Uni-MuMER-train.yaml"
OUTPUT_DIR = "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
RUN_UUID = uuid.uuid4().hex
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
if not DAGSHUB_TOKEN:
    raise RuntimeError("Kaggle Secret DAGSHUB_TOKEN is missing")

os.environ.pop("PYTHONPATH", None)
os.environ.update({
    "PROJECT_DIR": PROJECT_DIR,
    "CONDA_DIR": CONDA_DIR,
    "ENV_DIR": ENV_DIR,
    "PYTHON": PYTHON,
    "TRAIN_CONFIG": TRAIN_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_UUID": RUN_UUID,
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MLFLOW_FLATTEN_PARAMS": "TRUE",
    "MLFLOW_TAGS": json.dumps({
        "run_uuid": RUN_UUID,
        "source": "kaggle",
        "task": "sft",
        "dataset": "parquet_crohme_train",
    }),
})

print(f"Run UUID: {RUN_UUID}")
print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")

Run UUID: 3b0085a38ce4416d800464789c2405b0
MLflow: https://dagshub.com/NhatPot/test-unimer.mlflow


In [4]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

Python 3.10.20


In [5]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

217348e


Cloning into '/kaggle/working/test-unimer'...


In [6]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python --version
python -m pip install -q -r requirements.txt
python -m pip install -q -e train/LLaMA-Factory
python -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

Python 3.10.20
GPU: Tesla T4
MLflow: 3.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
prometheus-fastapi-instrumentator 8.0.2 requires starlette<2.0.0,>=1.0.0, but you have starlette 0.52.1 which is incompatible.


In [7]:
%%bash
# 5. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

🏃 View run righteous-skink-585 at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1/runs/23065cb569404ae19f8a97114031591b
🧪 View experiment at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1
DagsHub MLflow connection OK: https://dagshub.com/NhatPot/test-unimer.mlflow


In [8]:
%%bash
# 6. Training
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

echo "Python: $(command -v python)"
echo "Torchrun: $(command -v torchrun)"
MPLBACKEND=Agg llamafactory-cli train "$TRAIN_CONFIG" "run_name=uni-mumer-${RUN_UUID:0:8}"

Python: /kaggle/working/miniconda/envs/unimumer/bin/python
Torchrun: /kaggle/working/miniconda/envs/unimumer/bin/torchrun
[INFO|2026-06-23 12:34:48] llamafactory.launcher:143 >> Initializing 2 distributed tasks at: 127.0.0.1:54537
[WARNING|2026-06-23 12:34:57] llamafactory.hparams.parser:148 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-06-23 12:34:57] llamafactory.hparams.parser:143 >> Set `ddp_find_unused_parameters` to False in DDP training since LoRA is enabled.
[INFO|2026-06-23 12:34:57] llamafactory.hparams.parser:465 >> Process rank: 0, world size: 2, device: cuda:0, distributed training: True, compute dtype: torch.float16
[INFO|2026-06-23 12:34:58] llamafactory.hparams.parser:465 >> Process rank: 1, world size: 2, device: cuda:1, distributed training: True, compute dtype: torch.float16
[INFO|2026-06-23 12:35:01] llamafactory.data.loader:143 >> Loading dataset phxember/Uni-MuMER-Data...
training example:
input_ids:
[151644, 8948, 198, 2610, 525, 264

W0623 12:34:50.421000 393 site-packages/torch/distributed/run.py:792] 
W0623 12:34:50.421000 393 site-packages/torch/distributed/run.py:792] *****************************************
W0623 12:34:50.421000 393 site-packages/torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0623 12:34:50.421000 393 site-packages/torch/distributed/run.py:792] *****************************************
[INFO|tokenization_utils_base.py:2023] 2026-06-23 12:34:58,939 >> loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3/vocab.json
[INFO|tokenization_utils_base.py:2023] 2026-06-23 12:34:58,939 >> loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b82

In [9]:
%%bash
# 7. Upload và xác minh artifact
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$TRAIN_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR"

🏃 View run uni-mumer-3b0085a3 at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1/runs/46c9877fa9ca4f66bed168e93a0d30ff
🧪 View experiment at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1
Uploaded 2 artifact groups to DagsHub
Run: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1/runs/46c9877fa9ca4f66bed168e93a0d30ff
